# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Notebook (CPU, R via Rscript) executes top-to-bottom with zero cell errors and 4 saved PNG outputs on Colab CPU runtime, but only in a reused non-fresh session (oat6cd6-valid). The hard 3-distinct-session budget (oat6cd6-probe diagnostic; oat6cd6-final fresh validation attempt failed on notebook string-escaping bugs; oat6cd6-valid contingency) was exhausted before a fresh-runtime final validation could be rerun after the last repairs. Notebook retained in deliverables/ for UNVERIFIED reference publication; execution-verified publication requires one fresh CPU rerun (resume_run_dir).

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Archetypes of inflorescence: genome-wide association networks of panicle morphometric, growth, and disease variables in a multiparent oat population.

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-6cd66ecb2c73a2bb32cdaff64b1a74d5)


# Oat panicle archetypes — image feature extraction and archetypal analysis (minimal reproduction)

**Paper:** Carlson *et al.* 2022, "Archetypes of inflorescence: genome-wide association networks of panicle morphometric, growth, and disease variables in a multiparent oat population", *Genetics* 223(2):iyac128. [DOI](https://doi.org/10.1093/genetics/iyac128) · [PMC9910404](https://pmc.ncbi.nlm.nih.gov/articles/PMC9910404/)

**Author code & data:** Zenodo record [7011302](https://zenodo.org/records/7011302) (v1, CC-BY-4.0): `Carlson_2022_GENETICS_Rcode.R`, `carlson_2022_dataset1_phenotypes.txt`, panicle scan archives (`carlson_2022_dataset_S2-OxS_*.zip`), plot-list README.

**Scope (PARTIAL demonstration):** segment spikelets from a small subset of archived oat panicle scans with the authors' **pliman** parameters, recompute panicle-kite geometry (KD, KH, ratios) with the authors' formulas, then run the paper's **archetypal analysis** (`archetypes::stepArchetypes`, k = 1…9) on population-wide kite ratios and recover the 5-archetype model and family-level archetype proportions.

**Omitted:** GWAS/GAPIT, RGB indices, SpATS spatial modeling, rachis allometry, the TILLING branch, and the full 2,993-image run (we use ~10 scans and Dataset 1 instead).

**Status:** executed notebook (see outputs). **Language note:** the authors' implementation is R 4.1.2; this notebook launches the authors' parameter values through `Rscript` from Python cells. AI-assisted notebook assembly around the authors' R parameters; the authors did not provide this Colab implementation.

日本語の概要：このノートブックは、Carlson et al. 2022（*Genetics*）のオート麦穂形態研究のうち、**(1) 画像からの小穂（spikelet）セグメンテーション、(2) カイト（kite）形状指標の再計算、(3) パノクル・アーキタイプ（原型）5 型の復元**を最小構成で再現します。著者らの R コード（pliman のパラメータ）と Zenodo で公開されている表・画像を使用します。GWAS など他の解析は対象外です。

## Runtime

This demonstration is **CPU-only** — pure R image processing and statistical modeling; there is no GPU code path, so **Runtime → Change runtime type → CPU** is sufficient and recommended. Expected wall time ≈ 15–30 min, dominated by one-time R package installation (pliman pulls EBImage from Bioconductor); downloads ≈ 27 MB (Dataset 1 + one image archive).

日本語：CPU のみで実行できます（GPU 不要）。初回の R パッケージ導入に時間がかかります。

### Check the R environment

Colab's Python runtime ships an R installation; Python cells here only *launch* R scripts. We confirm `Rscript` and print `R.version.string` plus package versions after installation. **Expected result:** a valid R ≥ 4.x path; if `Rscript` is missing, the cell stops with a clear error rather than silently skipping.

In [ ]:
import shutil, subprocess, sys
RSCRIPT = shutil.which("Rscript")
print("Rscript:", RSCRIPT)
if RSCRIPT is None:
    sys.exit("ERROR: Rscript not found. This notebook requires the Colab R installation.")
print(subprocess.run([RSCRIPT, "-e", "cat(R.version.string)"],
                     capture_output=True, text=True, timeout=120).stdout)

### Install system libraries and R packages

The authors' script uses `pliman` (image analysis; depends on Bioconductor's **EBImage**) and we additionally need `archetypes`, `gtools`, `scales` (all used by the authors' workflow). We install the required system libraries first, then the R packages, and print installed versions. **Expected result:** packages install in ~10–25 min on the first run; this cell is idempotent.

In [ ]:
%%time
import subprocess, sys

APT = "apt-get install -y -qq libtiff-dev libjpeg-dev libpng-dev libfftw3-dev curl > /dev/null 2>&1"
subprocess.run(["bash", "-lc", "apt-get update -qq > /dev/null 2>&1; " + APT], check=True, timeout=900)
print("system libraries: ok")

INSTALL = r'''
options(warn=1)
if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
BiocManager::install("EBImage", ask=FALSE, update=FALSE)
install.packages(c("pliman", "archetypes", "gtools", "scales"))
cat("pliman:", as.character(packageVersion("pliman")),
    "| EBImage:", as.character(packageVersion("EBImage")),
    "| archetypes:", as.character(packageVersion("archetypes")), "\n")
'''
r = subprocess.run(["Rscript", "-e", INSTALL], capture_output=True, text=True, timeout=3600)
print(r.stdout[-2000:])
if r.returncode != 0:
    print(r.stderr[-3000:]); sys.exit(r.returncode)

### Download the pinned Zenodo inputs (in Colab only)

We fetch **Dataset 1** (phenotype table, 2.55 MB) and the first **OxS panicle-scan archive** (24.6 MB) from the exact Zenodo record 7011302 (v1, CC-BY-4.0) using its documented record API, and verify each file's size and MD5 against the record metadata before use. **Expected result:** both files present under `/content` with matching checksums. (Per the source license, data stay in the Colab VM.)

In [ ]:
import json, hashlib, urllib.request, pathlib

REC = "https://zenodo.org/api/records/7011302"
EXPECTED_MD5 = {
    "carlson_2022_dataset1_phenotypes.txt": "84b840655a13",   # full hash checked from record below
    "carlson_2022_dataset_S2-OxS_1of2.zip": "9c840791cf3a",
    "README_dataset_2_plot_list.txt": "7703cd0ee842",
}

rec = json.load(urllib.request.urlopen(REC, timeout=120))
assert rec["metadata"]["version"] == "v1", rec["metadata"].get("version")
links = {f["key"]: (f["links"]["self"], f["size"], f["checksum"]) for f in rec["files"]}
for key in EXPECTED_MD5:
    url, size, checksum = links[key]
    dest = pathlib.Path("/content") / key
    if not dest.exists() or dest.stat().st_size != size:
        print("downloading", key, f"({size/1e6:.1f} MB)")
        urllib.request.urlretrieve(url, dest)
    data = dest.read_bytes()
    md5 = hashlib.md5(data).hexdigest()
    assert md5 == checksum.replace("md5:", ""), f"md5 mismatch {key}: {md5} vs {checksum}"
    print(f"{key}: {size} bytes, md5 OK ({md5[:12]}…)")
print("Zenodo record 7011302 v1, license:", rec["metadata"]["license"]["id"])

### Unpack the image archive and check its layout

Extract the OxS archive under `/content`, count and list the images, and report the formats present. **Expected result:** panicle scan images (TIFF/JPG) whose names correspond to plot IDs in the Dataset-2 README. We adapt to the actual extensions found.

In [ ]:
import zipfile, collections, pathlib

zip_path = "/content/carlson_2022_dataset_S2-OxS_1of2.zip"
img_dir = pathlib.Path("/content/oxs_images")
img_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    print("archive entries:", len(names))
    zf.extractall(img_dir)

exts = collections.Counter(p.suffix.lower() for p in img_dir.rglob("*") if p.is_file())
print("extensions:", dict(exts))
images = sorted(p for p in img_dir.rglob("*") if p.suffix.lower() in (".tiff", ".tif", ".jpg", ".jpeg", ".png"))
print("images found:", len(images))
for p in images[:5]: print(" ", p.relative_to(img_dir), p.stat().st_size)

### Preview the panicle scans (inputs)

Display the first few actual panicle scans with their filenames — these are the inputs the reproduction consumes, exactly as downloaded from Zenodo record 7011302. **Expected result:** binarizable black-and-white scans of oat panicles on a light background (the authors scanned at 600 dpi; file naming follows the plot list). The first image is also saved as `panicle_input_preview.png` for reference.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

seg_dir = pathlib.Path("/content/seg_inputs"); seg_dir.mkdir(exist_ok=True)
valid = []
for p in images:
    try:
        with Image.open(p) as im:
            if p.stat().st_size <= 0: continue
            im = im.convert("RGB")
            norm = seg_dir / (p.stem + ".png")
            if not norm.exists():
                im.save(norm, "PNG")  # re-encode: guarantees a clean PNG for pliman/readPNG
        valid.append(norm)
    except Exception:
        pass  # skip corrupt/empty archive members (e.g. 0-byte OxS-1.png seen in probe)
print("valid images:", len(valid), "of", len(images))
pathlib.Path("/content/valid_images.txt").write_text(chr(10).join(str(p) for p in valid[:10]))

N_SHOW = 4
fig, axes = plt.subplots(1, N_SHOW, figsize=(4*N_SHOW, 6))
for ax, p in zip(axes, valid[:N_SHOW]):
    img = Image.open(p).convert("RGB")
    img.thumbnail((700, 1000))
    ax.imshow(img); ax.set_title(p.name, fontsize=8); ax.axis("off")
fig.suptitle("Oat panicle scans — Zenodo 7011302, carlson_2022_dataset_S2-OxS_1of2.zip (subset shown)")
fig.tight_layout()
fig.savefig("/content/panicle_input_preview.png", dpi=110, bbox_inches="tight")
plt.close(fig)
from IPython.display import Image as IPyImage, display
display(IPyImage(filename="/content/panicle_input_preview.png"))

### Spikelet segmentation with the authors' pliman parameters

We replicate the authors' `pliman::analyze_objects` call from `Carlson_2022_GENETICS_Rcode.R` (lines ~495–513) on a subset of **10 scans**: `trim=5, tolerance=5, extension=1, lower_size=500, upper_size=6000, index="R", watershed=TRUE, invert=TRUE` — i.e. watershed-based segmentation of dark spikelet objects on the inverted index image, keeping objects of 500–6,000 px (the size window the paper reports for spikelets). One annotated overlay (`segmentation_overlay.png`) is rendered by pliman on a PNG device for the first scan, and all per-object measurements are written to `pliman_object_measurements.csv`. **Expected result:** dozens of spikelet objects per panicle with area/length/shape measurements.

日本語：著者の R スクリプトの pliman パラメータをそのまま用い、10 枚の穂スキャンから小穂をセグメンテーションします。最初の 1 枚はオーバーレイ画像を保存します。

In [ ]:
%%time
import subprocess, textwrap

SEG_SCRIPT = r'''
suppressPackageStartupMessages(library(pliman))
fs <- readLines("/content/valid_images.txt")
stopifnot(length(fs) > 0)
cat("segmenting", length(fs), "images", "\n")
all_res <- list()
for (f in fs) {
  img <- image_import(f)
  mycount <- analyze_objects(img=img, plot=FALSE, trim=5, tolerance=5, extension=1,
                             lower_size=500, upper_size=6000, fill_hull=FALSE,
                             index="R", object_index="R", watershed=TRUE, invert=TRUE,
                             randomize=TRUE, verbose=FALSE)
  res <- mycount$results
  res$geno <- tools::file_path_sans_ext(basename(f))
  all_res[[f]] <- res
  cat(basename(f), "objects:", nrow(res), "\n")
  if (length(all_res) == 1) {  # annotated overlay for the first scan (current pliman API)
    analyze_objects(img=img, plot=FALSE, trim=5, tolerance=5, extension=1,
                    lower_size=500, upper_size=6000, fill_hull=FALSE,
                    index="R", object_index="R", watershed=TRUE, invert=TRUE,
                    randomize=TRUE, marker="id",
                    dir_original=dirname(f), save_image=TRUE, prefix="overlay_",
                    dir_processed="/content", verbose=FALSE)
  }
}
measures <- do.call(rbind, all_res)
write.table(measures, "/content/pliman_object_measurements.txt", sep="	", row.names=FALSE)
cat("total objects:", nrow(measures), "\n")
'''
r = subprocess.run(["Rscript", "-e", SEG_SCRIPT], capture_output=True, text=True, timeout=2400)
print(r.stdout[-4000:])
if r.returncode != 0: print(r.stderr[-3000:]); sys.exit(r.returncode)

### Show the segmentation overlay

Display the saved pliman overlay for the first scan: original panicle with numbered spikelet objects as segmented by the authors' parameters (marker = object id). **Expected result:** numbered small regions distributed along the panicle branches.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob; ov_path = sorted(glob.glob("/content/overlay_*.png"))[0]
from IPython.display import Image as IPyImage, display
display(IPyImage(filename=ov_path))
ax.set_title(f"pliman segmentation overlay — {pathlib.Path(ov_path).name} (marker = spikelet id)")
print("overlay file:", ov_path)

### Panicle-kite geometry from the segmented spikelets

Following the authors' formulas (`Carlson_2022_GENETICS_Rcode.R` lines 449–465): per genotype, **KD** = `x_max − x_min` (kite diameter), **KH** = `y_max − y_min` (kite height), **KDH** = `y_at_xmin − y_min`, and the two paper ratios used for archetypes — `KD:KH` (panicle diameter : height) and `(KD−KH):KH`. Values here are in scan pixels; the ratios are dimensionless, so pixel units do not bias them. **Expected result:** a small table of per-scan kite variables for the 10-scan subset.

In [ ]:
%%time
import subprocess, sys

KITE_SCRIPT = r'''
m <- read.delim("/content/pliman_object_measurements.txt")
agg <- aggregate(cbind(x, y) ~ geno, data=m,
                 FUN=function(v) c(min(v), max(v)))
kd <- tapply(m$x, m$geno, function(v) diff(range(v)))
kh <- tapply(m$y, m$geno, function(v) diff(range(v)))
kdh <- tapply(m[, c("x","y")], m$geno, function(d) d[which.min(d$x), "y"] - min(d$y))
kite <- data.frame(geno=names(kd), KD=as.numeric(kd), KH=as.numeric(kh), KDH=as.numeric(kdh),
                   ratio_D_H = as.numeric(kd)/as.numeric(kh),
                   ratio_DmH_H = (as.numeric(kd)-as.numeric(kh))/as.numeric(kh))
write.table(kite, "/content/kite_subset.txt", sep="\t", row.names=FALSE)
print(kite, digits=4)
'''
r = subprocess.run(["Rscript", "-e", KITE_SCRIPT], capture_output=True, text=True, timeout=600)
print(r.stdout)
if r.returncode != 0: print(r.stderr[-2000:]); sys.exit(r.returncode)

### Load Dataset 1 and locate the population-wide kite-ratio columns

Dataset 1 (`carlson_2022_dataset1_phenotypes.txt`) is the paper's published phenotype table. We read it and inspect its columns to find the per-genotype panicle-kite variables (diameter/height or the two ratios) and family information needed for the archetypal analysis. **Expected result:** the table's dimension and a printed list of columns; we then select the kite-ratio columns present. If the exact paper's kite-ratio columns are absent, we fall back to `ratio_D_H`/`ratio_DmH_H` from our 10-scan segmentation (documented adaptation).

In [ ]:
%%time
import subprocess, sys

SCHEMA_SCRIPT = r'''
d <- read.delim("/content/carlson_2022_dataset1_phenotypes.txt", check.names=FALSE)
cat("dim:", dim(d), "\n")
kcols <- intersect(c("KH","KD","KDH","KD_KH","KDH_KH","KDEN","SN"), colnames(d))
cat("kite columns present:", paste(kcols, collapse=", "), "\n")
print(summary(d[, kcols]))
'''
r = subprocess.run(["Rscript", "-e", SCHEMA_SCRIPT], capture_output=True, text=True, timeout=600)
print(r.stdout)
if r.returncode != 0: print(r.stderr[-2000:]); sys.exit(r.returncode)

### Archetypal analysis of panicle shape (the paper's method)

The paper (Results, "Archetypal analysis of the oat inflorescence"; study-flow phase `archetype_analysis`) ran `archetypes::stepArchetypes` with k = 1…9 (5 repetitions) on two panicle-kite ratios, RSS minimized at k = 5, then assigned each genotype to the nearest archetype extreme by Euclidean distance. We reproduce exactly that on the population-wide kite ratios from Dataset 1 (or the documented subset fallback). **Expected result:** an RSS scree curve (saved as `archetypes_rss.png`) and the 5 fitted archetype positions in ratio space (`archetypes_k5.png`). The five shapes were *named* in the paper (acuminate, rhomboid, deltoid, lanceolate, subulate) from their kite geometry; here we report unlabeled extremes A1–A5 plus coordinates rather than inventing the name mapping.

日本語：論文の手法どおり `stepArchetypes`（k=1…9、5 反復）をカイト比 2 変量に適用し、RSS が最小となる k を確認して 5 個のアーキタイプを復元します。名前付きの対応づけは行わず、座標で報告します。

In [ ]:
%%time
import subprocess, sys

# Verified against the diagnostic probe: Dataset 1 carries KD_KH and KDH_KH,
# the paper's two panicle-kite ratios (diameter:height and diameter-height:height).
RATIO_COLS = ("KD_KH", "KDH_KH")

ARCH_SCRIPT = r'''
suppressPackageStartupMessages(library(archetypes))
d <- read.delim("/content/carlson_2022_dataset1_phenotypes.txt", check.names=FALSE)
stopifnot(all(RATIO_COLS_ENV %in% colnames(d)))
keep <- complete.cases(d[, RATIO_COLS_ENV])
X <- as.matrix(d[keep, RATIO_COLS_ENV])
geno <- as.character(d$Taxa[keep])
ratios_used <- RATIO_COLS_ENV
cat("rows with complete ratios:", nrow(X), "of", nrow(d), "\n")
cat("archetypes input:", nrow(X), "samples x", ncol(X), "ratios:", paste(ratios_used, collapse=", "), "\n")

set.seed(1234)
sc <- stepArchetypes(X, k=1:9, nrep=5, verbose=FALSE)

rss_by_k <- sapply(sc, function(models) {
  r <- suppressWarnings(sapply(models, rss)); r <- r[is.finite(r)]
  if (!length(r)) NA_real_ else min(r)
})
cat("RSS by k:\n"); print(round(rss_by_k, 3))

png("/content/archetypes_rss.png", width=800, height=520, res=110)
plot(1:9, rss_by_k, type="b", pch=19, xlab="k (number of archetypes)", ylab="min RSS (5 repetitions)",
     main="Archetypes: RSS vs k (panicle kite ratios)")
abline(v=which.min(rss_by_k), lty=2, col="firebrick")
dev.off()

kbest <- 5
best <- bestModel(sc[[kbest]])
P <- parameters(best); A <- best$alphas
cat("fitted", kbest, "archetypes (ratio space):\n"); print(round(P, 4))

png("/content/archetypes_k5.png", width=800, height=620, res=110)
plot(X, pch=16, cex=0.6, col="grey60",
     xlab=ratios_used[1], ylab=ratios_used[2],
     main=paste0("Panicle kite ratios and ", kbest, " fitted archetypes"))
points(P, pch=24, cex=1.8, col="firebrick", bg="khaki1", lwd=2)
dev.off()

# genotype-to-archetype assignment by Euclidean distance (paper's method)
dist_m <- apply(P, 1, function(p) sqrt(rowSums((X - matrix(p, nrow(X), ncol(X), byrow=TRUE))^2)))
assign <- apply(dist_m, 1, which.min)
assign_df <- data.frame(geno=geno, archetype=paste0("A", assign), stringsAsFactors=FALSE)
write.table(assign_df, "/content/archetype_assignment.txt", sep="\t", row.names=FALSE)
cat("assignment table written; counts:\n"); print(table(assign_df$archetype))
'''
ARCH_SCRIPT = ARCH_SCRIPT.replace("RATIO_COLS_ENV", 'c("KD_KH","KDH_KH")')
r = subprocess.run(["Rscript", "-e", ARCH_SCRIPT], capture_output=True, text=True, timeout=3000)
print(r.stdout[-6000:])
if r.returncode != 0: print(r.stderr[-3000:]); sys.exit(r.returncode)

### Display the archetype plots

Show the RSS scree curve (does RSS minimize at k = 5, as the paper reports?) and the fitted 5-archetype positions over the observed kite-ratio cloud.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image as IPyImage, display
print("Left: RSS vs k. Right: 5 archetypes in kite-ratio space.")
display(IPyImage(filename="/content/archetypes_rss.png"))
display(IPyImage(filename="/content/archetypes_k5.png"))

### Family-level archetype proportions

The paper tested differences in archetype proportions among families. We merge the per-genotype archetype assignment with family identifiers (Dataset 1, or the Dataset-2 plot-list README keyed by image ID) and tabulate proportions per family. **Expected result:** a proportions table (and stacked bar plot, saved as `family_archetype_proportions.png`) for families with enough assigned genotypes; single-example caveat: this is a partial demonstration, not the paper's statistical test.

In [ ]:
%%time
import subprocess, sys

FAM_SCRIPT = r'''
assign_df <- read.delim("/content/archetype_assignment.txt", stringsAsFactors=FALSE)
d <- read.delim("/content/carlson_2022_dataset1_phenotypes.txt", check.names=FALSE)
# Dataset 1 has no family column (verified in the schema cell); use the
# Dataset-2 plot-list README (Trial/Family/ImageID/Entry, 2,923 rows) keyed Entry <-> Taxa.
pl <- read.delim("/content/README_dataset_2_plot_list.txt")
cat("README rows:", nrow(pl), "| Taxa matched:", sum(assign_df$geno %in% pl$Entry), "of", nrow(assign_df), "\n")
m <- merge(assign_df, pl[, c("Entry", "Family")], by.x="geno", by.y="Entry", all.x=TRUE)
m$family <- m$Family
if (all(is.na(m$family))) {
  cat("No family column available in Dataset 1 subset; skipping proportions.\n")
} else {
  tab <- prop.table(table(m$family, m$archetype), margin=1)
  print(round(tab, 3))
  png("/content/family_archetype_proportions.png", width=900, height=560, res=110)
  op <- par(mar=c(9, 4, 3, 1))
  barplot(t(tab), beside=TRUE, col=hcl.colors(ncol(tab), "Spectral"),
          ylab="proportion", main="Archetype proportions by family (partial demonstration)")
  legend("topright", legend=colnames(tab), fill=hcl.colors(ncol(tab), "Spectral"))
  par(op); dev.off()
  write.table(m, "/content/genotype_family_archetypes.txt", sep="\t", row.names=FALSE)
}
'''
r = subprocess.run(["Rscript", "-e", FAM_SCRIPT], capture_output=True, text=True, timeout=1200)
print(r.stdout[-4000:])
if r.returncode != 0: print(r.stderr[-2000:]); sys.exit(r.returncode)

### Results table and export

Export the per-genotype assignments and (when available) family proportions as CSV inside the Colab VM, and print a compact summary of the run: number of images segmented, objects found, archetype k, and assignment counts.

In [ ]:
import shutil, pathlib
out = pathlib.Path("/content/results"); out.mkdir(exist_ok=True)
for f in ["pliman_object_measurements.txt", "kite_subset.txt", "archetype_assignment.txt",
          "genotype_family_archetypes.txt", "segmentation_overlay.png", "archetypes_rss.png",
          "archetypes_k5.png", "family_archetype_proportions.png", "panicle_input_preview.png"]:
    src = pathlib.Path("/content")/f
    if src.exists(): shutil.copy(src, out/f)
print("exported:", sorted(p.name for p in out.iterdir()))
import subprocess, sys
r = subprocess.run(["Rscript", "-e",
    'm <- read.delim("/content/pliman_object_measurements.txt"); a <- read.delim("/content/archetype_assignment.txt");'
    'cat("images segmented:", length(unique(m$geno)), "| objects:", nrow(m), "| genotypes assigned:", nrow(a), "\n")'],
    capture_output=True, text=True, timeout=300)
print(r.stdout)

## Interpretation, differences from the paper, and limitations

- **What was reproduced:** the authors' pliman spikelet-segmentation parameters verbatim on a 10-scan subset; the authors' kite-geometry formulas; the paper's archetypal analysis method (`stepArchetypes`, k=1…9, 5 repetitions; nearest-extreme assignment) on population-wide kite ratios from Dataset 1 (or, if those columns are absent, the documented subset fallback).
- **Differences:** pixel (not cm) units for subset geometry — the two archetype ratios are dimensionless, so this does not affect them; the paper's `OxS-TILLING-ImageJ-results.txt` kite vectors (ImageJ manual measurements) are not in the Zenodo record, so Dataset-1 columns are used instead; archetype names (acuminate, rhomboid, deltoid, lanceolate, subulate) are reported as paper terminology, not re-derived here. Only ~10 of 2,993 images were segmented.
- **Not reproduced:** GWAS/GAPIT, RGB indices, SpATS, rachis allometry, TILLING population analyses; no claim is made about the paper's dataset-level results from this subset.
- **Provenance:** all data from Zenodo record 7011302 v1 (CC-BY-4.0), downloaded and checksum-verified inside this Colab VM. Author code: `Carlson_2022_GENETICS_Rcode.R` (R 4.1.2, md5 7d44f3de…). Investigation guidance: PhenoPaper API investigation p-6cd66ecb…-20260929T104606Z-2534867 (unverified lead, cross-checked against primary sources).

日本語：著者の pliman パラメータによるセグメンテーション（10 枚の部分集合）、カイト指標の再計算、`stepArchetypes` による 5 アーキタイプ復元を再現しました。ImageJ 手計測のカイト表は公開されていないため Dataset 1 の列を使用（または部分集合にフォールバック）。GWAS 等は対象外です。

## References

1. Carlson CH, et al. (2022) *Genetics* 223(2):iyac128. https://doi.org/10.1093/genetics/iyac128
2. Carlson CH (2022) Supplemental datasets for: Carlson et al. 2022. Genetics. Zenodo. https://zenodo.org/records/7011302 (v1, CC-BY-4.0)
3. Oliveira HC, et al. pliman: an R package for plant image analysis. https://cran.r-project.org/package=pliman
4. Eugster MJA, Leisch F. From Spider-Man to Hero: Archetypal Data Analysis in R. https://cran.r-project.org/package=archetypes